## Deep Learning Basics with PyTorch
### Part III - Supervised Deep Learning in Practice
## Chapter 10 - Improving Training

This chapter introduces a few practical tools for controlling training: weight decay, dropout, early stopping, and learning-rate schedules.
The goal is to watch how train and validation behavior changes, not to build a larger model.


In [ ]:
# !pip -q install torch numpy matplotlib
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch import nn

plt.style.use('seaborn-v0_8')
%config InlineBackend.figure_format = 'retina'


def set_seed(seed=0):
    torch.manual_seed(seed)
    np.random.seed(seed)

**Data helpers** — `make_two_moons` generates the two-moons classification dataset; `split_train_val` carves out stratified train and validation splits.

In [ ]:
def make_two_moons(n_samples=1000, noise=0.35, seed=0):
    generator = torch.Generator().manual_seed(seed)
    n_top = n_samples // 2
    n_bottom = n_samples - n_top

    theta_top = torch.linspace(0, torch.pi, n_top)
    theta_bottom = torch.linspace(0, torch.pi, n_bottom)

    top = torch.stack([torch.cos(theta_top), torch.sin(theta_top)], dim=1)
    bottom = torch.stack(
        [1 - torch.cos(theta_bottom), -torch.sin(theta_bottom) - 0.5], dim=1
    )

    X = torch.cat([top, bottom], dim=0)
    y = torch.cat(
        [
            torch.zeros(n_top, dtype=torch.long),
            torch.ones(n_bottom, dtype=torch.long),
        ]
    )

    X = X + noise * torch.randn(X.shape, generator=generator)
    return X, y


def split_train_val(X, y, train_fraction=0.35, val_fraction=0.20, seed=42):
    generator = torch.Generator().manual_seed(seed)
    train_idx_parts = []
    val_idx_parts = []

    for label in [0, 1]:
        idx = torch.where(y == label)[0]
        idx = idx[torch.randperm(len(idx), generator=generator)]
        n_train = int(len(idx) * train_fraction)
        n_val = int(len(idx) * val_fraction)
        train_idx_parts.append(idx[:n_train])
        val_idx_parts.append(idx[n_train:n_train + n_val])

    train_idx = torch.cat(train_idx_parts)
    val_idx = torch.cat(val_idx_parts)
    train_idx = train_idx[torch.randperm(len(train_idx), generator=generator)]
    val_idx = val_idx[torch.randperm(len(val_idx), generator=generator)]
    return X[train_idx], X[val_idx], y[train_idx], y[val_idx]

**Model and training helpers** — `build_model` constructs a two-layer MLP with optional dropout; `run_curves` trains it and returns per-epoch train and validation losses.

In [ ]:
def build_model(hidden=256, dropout_p=0.0):
    layers = [nn.Linear(2, hidden), nn.ReLU()]
    if dropout_p > 0:
        layers.append(nn.Dropout(dropout_p))
    layers.append(nn.Linear(hidden, 2))
    return nn.Sequential(*layers)


def run_curves(X_tr, y_tr, X_va, y_va, *, weight_decay=0.0, dropout_p=0.0,
               epochs=100, lr=5e-3, hidden=256, seed=0, scheduler=None):
    set_seed(seed)
    model = build_model(hidden=hidden, dropout_p=dropout_p)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = scheduler(opt) if scheduler is not None else None
    train_losses, val_losses = [], []

    for _ in range(epochs):
        model.train()
        logits = model(X_tr)
        train_loss = F.cross_entropy(logits, y_tr)
        opt.zero_grad()
        train_loss.backward()
        opt.step()
        if sched is not None:
            sched.step()

        model.eval()
        with torch.no_grad():
            val_loss = F.cross_entropy(model(X_va), y_va)

        train_losses.append(float(train_loss.detach()))
        val_losses.append(float(val_loss))

    return train_losses, val_losses

**Dataset creation** — generates the two-moons dataset and splits it into train and validation tensors used throughout the chapter.

In [ ]:
set_seed(0)
X, y = make_two_moons(n_samples=1000, noise=0.35, seed=0)
X_tr, X_va, y_tr, y_va = split_train_val(X, y, train_fraction=0.35, val_fraction=0.20, seed=42)

print('Train split:', X_tr.shape, y_tr.shape)
print('Validation split:', X_va.shape, y_va.shape)

## Train/val curves with and without weight decay

Notice the gap between the training curve and the validation curve. Weight decay adds a penalty on large weights, which can reduce overfitting on this small training split.


In [ ]:
tr0, va0 = run_curves(X_tr, y_tr, X_va, y_va, weight_decay=0.0, dropout_p=0.0, epochs=100, lr=5e-3, hidden=256, seed=0)
tr1, va1 = run_curves(X_tr, y_tr, X_va, y_va, weight_decay=5e-3, dropout_p=0.0, epochs=100, lr=5e-3, hidden=256, seed=0)

epochs = range(1, len(tr0) + 1)
fig, ax = plt.subplots(1, 2, figsize=(8, 3))

ax[0].plot(epochs, tr0, label='train')
ax[0].plot(epochs, va0, label='val')
ax[0].set_title('No Weight Decay')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('Loss')
ax[0].legend(frameon=False)

ax[1].plot(epochs, tr1, label='train')
ax[1].plot(epochs, va1, label='val')
ax[1].set_title('Weight Decay = 0.005')
ax[1].set_xlabel('Epoch')
ax[1].legend(frameon=False)

plt.tight_layout()
plt.show()


## Dropout comparison

Dropout randomly removes part of the hidden representation during training. Compare the validation curves and notice how strong dropout changes the shape of learning.


In [ ]:
tr0, v0 = run_curves(X_tr, y_tr, X_va, y_va, weight_decay=0.0, dropout_p=0.0, epochs=100, lr=5e-3, hidden=256, seed=0)
tr1, v1 = run_curves(X_tr, y_tr, X_va, y_va, weight_decay=0.0, dropout_p=0.6, epochs=100, lr=5e-3, hidden=256, seed=0)

epochs_range = range(1, len(v0) + 1)
plt.figure(figsize=(6, 3))
plt.plot(epochs_range, tr0, linestyle='--', color='tab:blue',   label='train p=0.0')
plt.plot(epochs_range, v0,  linestyle='-',  color='tab:blue',   label='val   p=0.0')
plt.plot(epochs_range, tr1, linestyle='--', color='tab:orange', label='train p=0.6')
plt.plot(epochs_range, v1,  linestyle='-',  color='tab:orange', label='val   p=0.6')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

## Early stopping (keep best validation model)

Early stopping watches validation loss after each epoch.
The notebook stores the best-performing model state, resets a wait counter when validation improves, and stops once that wait counter reaches the patience limit.


In [ ]:
def train_with_early_stopping(X_tr, y_tr, X_va, y_va, *, epochs=120, patience=15, lr=5e-3, weight_decay=1e-3, hidden=32, seed=0):
    set_seed(seed)
    model = build_model(hidden=hidden, dropout_p=0.0)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()

    train_losses, val_losses = [], []
    best_state = None
    best_val = float('inf')
    best_epoch = 0
    wait = 0
    stop_epoch = epochs

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = loss_fn(model(X_tr), y_tr)
        opt.zero_grad()
        train_loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_va), y_va)

        train_losses.append(float(train_loss.detach()))
        val_losses.append(float(val_loss))

        if float(val_loss) < best_val - 1e-4:
            best_val = float(val_loss)
            best_epoch = epoch
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                stop_epoch = epoch
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, {
        'train': train_losses,
        'val': val_losses,
        'best_val': best_val,
        'best_epoch': best_epoch,
        'stop_epoch': stop_epoch,
        'patience': patience,
    }


best_model, early = train_with_early_stopping(X_tr, y_tr, X_va, y_va, epochs=120, patience=15, lr=5e-3, weight_decay=1e-3, hidden=32, seed=0)

epochs = range(1, len(early['train']) + 1)
plt.figure(figsize=(6.4, 3.2))
plt.plot(epochs, early['train'], label='train')
plt.plot(epochs, early['val'], label='val')
plt.axvline(early['best_epoch'], color='gray', linestyle='--', label='best epoch')
plt.axvline(early['stop_epoch'], color='tab:red', linestyle=':', label='stop epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

print(f"Best validation loss: {early['best_val']:.3f}")
print(f"Best epoch: {early['best_epoch']}")
print(f"Stopped at epoch: {early['stop_epoch']} with patience = {early['patience']}")


## Learning-rate schedules — StepLR vs CosineAnnealingLR

A learning-rate schedule changes the optimizer step size over time.

- **StepLR** (`step_size=5, gamma=0.5`): multiplies the LR by `gamma` every `step_size` epochs — a discrete staircase drop.
- **CosineAnnealingLR** (`T_max=20`): decays the LR smoothly from its initial value to near zero following a half-cosine curve.

Both schedulers are applied to the same model and dataset so the resulting validation curves are directly comparable.

In [ ]:
import functools

def make_step(opt):
    return torch.optim.lr_scheduler.StepLR(opt, step_size=5, gamma=0.5)

def make_cosine(opt):
    return torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=20)

_, va_step   = run_curves(X_tr, y_tr, X_va, y_va, epochs=20, lr=5e-3, hidden=256, seed=0,
                          scheduler=make_step)
_, va_cosine = run_curves(X_tr, y_tr, X_va, y_va, epochs=20, lr=5e-3, hidden=256, seed=0,
                          scheduler=make_cosine)

ep = range(1, 21)
plt.figure(figsize=(6, 3))
plt.plot(ep, va_step,   label='StepLR (step=5, γ=0.5)')
plt.plot(ep, va_cosine, label='CosineAnnealingLR (T_max=20)')
plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

print("Takeaway: CosineAnnealingLR typically yields a smoother validation curve than StepLR "
      "because its gradual decay avoids the abrupt loss spikes that can follow a sudden step drop.")

## Chapter 10 Summary

- Weight decay and dropout are two common ways to regularize a model and reduce overfitting.
- Early stopping uses validation monitoring to keep the best model instead of always taking the last epoch.
- A patience value controls how long training continues without improvement.
- Learning-rate schedules change the optimizer step size over time and can make training more controlled.
